In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

Computes variance of samples for a single instance and aggregates over clusters

This code produced the plots in 3.4a,b and 3.5b,c 

# Variance of Predictions

In [ ]:
test_df=pd.read_csv('/opig-shared/users/lina4783/structures_final/test_meta.csv')
len(test_df)

In [ ]:
test_rmsds_summary=pd.read_csv('/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_sample_imgt/struc_pred_metrics_test_summary.csv')
len(test_rmsds_summary)

In [ ]:
test_rmsds_summary.head()

In [ ]:
test_rmsds_summary.iloc[0]

In [ ]:
import matplotlib.ticker as ticker

cdrh3_vars = np.array(test_rmsds_summary['H_cdr3_var'])

fig, ax = plt.subplots()

ax.violinplot(cdrh3_vars)
ax.set_yscale('log')

# Show decimal values instead of 10^n
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:g}'))
ax.yaxis.set_minor_formatter(ticker.NullFormatter())  # optional

plt.title('Variance of CDRH3 RMSD to GT Structure Over 10 Samples')
plt.ylabel('Angstroms')
plt.show()


In [ ]:
similarities_df=pd.read_csv('/opig-shared/users/lina4783/cdrh3_similarity_out/train_test_concat_CDR/nearest_train_per_test.csv')
len(similarities_df)

In [ ]:
merged_df=pd.merge(similarities_df,test_rmsds_summary,left_on='test_id',right_on='pdb_name',how='left')

In [ ]:
merged_df.head()

In [ ]:
# Randomly sample 200 rows each time 
plot_df = merged_df.sample(n=min(100, len(merged_df)))

yerr = np.sqrt(plot_df['H_cdr3_var'])

plt.figure(figsize=(6, 4.5))
plt.errorbar(
    plot_df['best_identity'],
    plot_df['H_cdr3'],
    yerr=yerr,
    fmt='o',
    ecolor='gray',
    elinewidth=1,
    capsize=2,
    alpha=0.7
)

plt.xlabel('% CDRH3 Sequence Similarity')
plt.ylabel('CDRH3 RMSD')
plt.title('CDRH3 similarity vs CDRH3 RMSD n=50')
plt.tight_layout()
plt.show()

In [ ]:
x=np.log(merged_df['H_cdr3'])
y=np.log(merged_df['H_cdr3_var'])

m, b = np.polyfit(x, y, 1)

plt.scatter(x,y)
plt.xlabel('CDRH3 RMSD')
plt.ylabel('CDRH3 RMSD Variance')
plt.plot(x, m * x + b, color="red",
         label=f"y = {m:.2f}x + {b:.2f}")
plt.legend()
plt.show()


In [ ]:
merged_df=pd.read_csv('/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_ckpt_5139_imgt/struc_pred_metrics_test_summary.csv')
y = np.log(merged_df['H_cdr3'])
x = np.log(merged_df['H_cdr3_var'])

# Fit line
m, b = np.polyfit(x, y, 1)

# Predicted values
y_pred = m * x + b

# Compute R²
ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - np.mean(y))**2)
r_sq = 1 - ss_res / ss_tot

plt.scatter(x, y)
plt.plot(x, y_pred, color="red",
         label=f"y = {m:.2f}x + {b:.2f}")

plt.xlabel("log CDRH3 RMSD Variance")
plt.ylabel("log CDRH3 RMSD")
plt.title(f"log CDRH3 vs log CDRH3 Variance(n=10) $R^2$={r_sq:.3f}")

plt.legend()
plt.show()

In [ ]:
x = merged_df['test_seq'].str.len()
y = np.log(merged_df['H_cdr3_var'])

# Fit line
m, b = np.polyfit(x, y, 1)

# Predicted values
y_pred = m * x + b

# Compute R²
ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - np.mean(y))**2)
r_sq = 1 - ss_res / ss_tot

plt.scatter(x, y)
plt.plot(x, y_pred, color="red",
         label=f"y = {m:.2f}x + {b:.2f}")

plt.xlabel("CDRH3 Sequence Length")
plt.ylabel("log CDRH3 RMSD Variance")
plt.title(f"CDRH3 seq length vs log CDRH3 Variance(n=10) $R^2$={r_sq:.3f}")

plt.legend()
plt.show()

In [ ]:
rmsds_df=pd.read_csv('/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_ckpt_5139_imgt/struc_pred_metrics_test_all_samples.csv')

#add cluster numbers to rmsds_df (from test_meta)
test_meta=pd.read_csv('/opig-shared/users/lina4783/structures_final/test_meta.csv')
cluster_map=test_meta.set_index('pdb_name')['cluster_ids']
rmsds_df['cluster_id']=rmsds_df['pdb_name'].map(cluster_map)
rmsds_df.head()




In [ ]:
len(rmsds_df)

In [ ]:
rmsds_df=rmsds_df[rmsds_df['status']=='ok']
rmsds_df.cluster_id.value_counts()

In [ ]:
len(rmsds_df[rmsds_df['cluster_id']==0])/10

In [ ]:
test_meta[test_meta['cluster_ids']==0]

In [ ]:
order = (
    rmsds_df.groupby("pdb_name")["H_cdr3"]
    .mean()
    .sort_values()
    .index
)

cluster_map = rmsds_df.groupby("pdb_name")["cluster_id"].first()
palette = {
    pdb: sns.color_palette("tab20", len(rmsds_df["cluster_id"].unique()))[
        list(sorted(rmsds_df["cluster_id"].unique())).index(cluster_map[pdb])
    ]
    for pdb in order
}

plt.figure(figsize=(24,6))

sns.boxplot(
    data=rmsds_df,
    x="pdb_name",
    y="H_cdr3",
    order=order,
    palette=palette,
    linewidth=1,
    fliersize=2,
)

plt.xticks(rotation=90)
plt.xlabel("PDB id")
plt.ylabel("CDRH3 RMSD (n=10)")
plt.tight_layout()
plt.show()

In [ ]:


def plot_sampled_clusters_boxplot(
    rmsds_df,
    n_clusters=10,
    random_state=0,
    pdb_col="pdb_name",
    rmsd_col="H_cdr3",
    cluster_col="cluster_id",
):
    # Pick whole clusters at random
    rng = np.random.default_rng(random_state)
    unique_clusters = rmsds_df[cluster_col].dropna().unique()
    n_clusters = min(n_clusters, len(unique_clusters))
    sampled_clusters = rng.choice(unique_clusters, size=n_clusters, replace=False)

    # Keep only rows from the sampled clusters
    plot_df = rmsds_df[rmsds_df[cluster_col].isin(sampled_clusters)].copy()

    # Order pdbs by mean RMSD within the sampled subset
    order = (
        plot_df.groupby(pdb_col)[rmsd_col]
        .mean()
        .sort_values()
        .index
    )

    # Map each pdb to its cluster, and each cluster to a color
    cluster_map = plot_df.groupby(pdb_col)[cluster_col].first()
    cluster_list = list(pd.unique(plot_df[cluster_col]))
    palette = sns.color_palette("tab20", n_colors=len(cluster_list))
    cluster_colors = dict(zip(cluster_list, palette))
    box_colors = [cluster_colors[cluster_map[pdb]] for pdb in order]

    fig, ax = plt.subplots(figsize=(24, 6))

    bp = ax.boxplot(
        [plot_df.loc[plot_df[pdb_col] == pdb, rmsd_col] for pdb in order],
        patch_artist=True,
        widths=0.6,
        showfliers=True,
        flierprops=dict(marker="o", markersize=2, alpha=0.4),
    )

    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_edgecolor("black")
        patch.set_alpha(0.9)

    for whisker in bp["whiskers"]:
        whisker.set(color="gray", linewidth=1)

    for cap in bp["caps"]:
        cap.set(color="gray", linewidth=1)

    for median in bp["medians"]:
        median.set(color="black", linewidth=1.5)

    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels(order, rotation=90, fontsize=8)
    ax.set_xlabel("PDB id")
    ax.set_ylabel("CDRH3 RMSD (n=10)")
    ax.set_title(f"CDRH3 RMSD Distribution Acrross Samples (random {n_clusters} clusters)")

    legend_handles = [
        Patch(facecolor=cluster_colors[c], edgecolor="black", label=str(c))
        for c in cluster_list
    ]
    ax.legend(handles=legend_handles, title="Cluster",
              bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

    return sampled_clusters, plot_df

In [ ]:
sampled_clusters, plot_df = plot_sampled_clusters_boxplot(
    rmsds_df,
    n_clusters=15,
    random_state=3
)

In [ ]:
sampled_clusters, plot_df = plot_sampled_clusters_boxplot(
    rmsds_df,
    n_clusters=15,
    random_state=0
)

In [ ]:
#get cluster variance of mean and variance of variance (for clusters with >2 pdb ids)
cluster_sizes=rmsds_df.groupby('cluster_id').size()
cluster_sizes=cluster_sizes[cluster_sizes>2]
cluster_sizes.head()
rmsds_df['cluster_size']=np.int64(rmsds_df['cluster_id'].map(cluster_sizes)/10)



In [ ]:
rmsds_df['cluster_size'].value_counts()

In [ ]:
usable_rmsds_df=rmsds_df[rmsds_df['cluster_size']>2]


# Step 1: summarize each pdb
pdb_summary = (
    usable_rmsds_df
    .groupby(["cluster_id", "pdb_name"])
    .agg(
        mean_rmsd=("H_cdr3", "mean"),
        var_rmsd=("H_cdr3", "std")
    )
    .reset_index()
)

# Step 2: summarize each cluster
cluster_summary = (
    pdb_summary
    .groupby("cluster_id")
    .agg(
        n_pdbs=("pdb_name", "nunique"),
        mean_of_means=("mean_rmsd", "mean"),
        var_of_means=("mean_rmsd", "var"),
        mean_of_variances=("var_rmsd", "mean"),
        var_of_variances=("var_rmsd", "var"),
    )
    .reset_index()
)

cluster_summary

In [ ]:
low_rmsd_clusters=cluster_summary[cluster_summary['mean_of_means']<2.5]
low_rmsd_clusters


In [ ]:
low_rmsd_clusters_df=test_meta[test_meta['cluster_ids'].isin(low_rmsd_clusters.cluster_id.unique())]

low_rmsd_clusters_df.to_csv('/opig-shared/users/lina4783/structures_final/train_meta_low_rmsd_clusters_nonsumbsample.csv',index=False)

In [ ]:
high_rmsd_clusters=cluster_summary[cluster_summary['mean_of_means']>6]
high_rmsd_clusters.cluster_id.unique()



In [ ]:
#get test_meta datapoints of high rmsd clusters
high_rmsd_clusters_df=test_meta[test_meta['cluster_ids'].isin(high_rmsd_clusters.cluster_id.unique())]
high_rmsd_clusters_df.head()

high_rmsd_clusters_df.to_csv('/opig-shared/users/lina4783/structures_final/train_meta_high_rmsd_clusters.csv')


In [ ]:
len(rmsds_df)

In [ ]:
print(cluster_order.values)

In [ ]:

# order clusters by average PDB mean
cluster_order = (
    pdb_summary
    .groupby("cluster_id")["mean_rmsd"]
    .mean()
    .sort_values()
    .index
)

plt.figure(figsize=(18,6))

sns.boxplot(
    data=pdb_summary,
    x="cluster_id",
    y="mean_rmsd",
    order=cluster_order,
    color="tomato",
    fliersize=2,
)

plt.axhline(y=2.5, color='red', linestyle='--')

plt.xticks(rotation=90)
plt.xlabel("Cluster")
plt.ylabel("Mean CDRH3 RMSD (n=10 per PDB ID) (Angstroms)")
plt.title("Performance (CDRH3 RMSD) by Cluster")
plt.ylim(0,14)

plt.tight_layout()
plt.show()

In [ ]:
#run the same thing for ABB3 on its training set (they also have a training set)(goes in report)
#can start from current checkpoint with new loss weights (turn on BB)
#take some of bad ones and overlay in pymol and send to ody

In [ ]:
# order clusters by average within-PDB variance
cluster_order = (
    pdb_summary
    .groupby("cluster_id")["var_rmsd"]
    .mean()
    .sort_values()
    .index
)

plt.figure(figsize=(18,6))

sns.boxplot(
    data=pdb_summary,
    x="cluster_id",
    y="var_rmsd",
    order=cluster_order,
    color="steelblue",
    fliersize=2,
)

#plt.axhline(y=2.5, color='red', linestyle='--')

plt.xticks(rotation=90)
plt.xlabel("Cluster")
plt.ylabel("Standard Deviation of CDRH3 RMSD (n=10 per PDB ID) (Angstroms)")
plt.title("Distribution of CDRH3 RMSD standard deviations per sample, broken down by cluster")
plt.ylim(0,3)

plt.tight_layout()
plt.show()